In [ ]:
from forestkernel import ForestKernel
from sklearn.model_selection import train_test_split
from dataset import dataprep


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Read in the data and normalize

In [ ]:
# TODO: Test with NumPy and Pandas, ALL METHODS and perhaps combos of each type. Problem with categorical data in Pandas?
# We should probably modify RFGAP to handle categorical data in the form of strings.

In [ ]:
seed = 42

kernel_method = 'gap'
model_type = 'rf'

force_symmetric = False
force_nonzero_diag = False
normalize_diagonal = False
oob_score = True
max_samples = None

n_estimators = 100
max_depth = 0 if model_type == 'xgb' else None
verbose=1
n_jobs = -1



test_size=0.2

# data = pd.read_parquet('./data/sign_mnist_train.parquet')
# data = pd.read_parquet('./data/tv_news_combined.parquet')
data = pd.read_csv('./data/iris.csv')


x, y   = dataprep(data)

n_samples = x.shape[0]
n_features = x.shape[1]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = test_size, random_state = 42)

print(f"Train samples: {x_train.shape[0]}, Test samples: {x_test.shape[0]}")
print(f"Number of features: {x_train.shape[1]}")
print(max_depth)

## Train the RF Model

In [ ]:
rf = ForestKernel(y = y_train, kernel_method = kernel_method, model_type=model_type,
           oob_score = True,
           max_samples = max_samples,
           force_nonzero_diag = force_nonzero_diag,
           random_state = seed,
           n_jobs = n_jobs,
           verbose=verbose,

           n_estimators = n_estimators,
           max_depth = max_depth,

           #XGBoost parameters
        #    device='cuda',
        

           # LigthGBM parameters
            #    num_leaves = 1023,
            #    num_leaves = 131072,
            #    min_data_in_leaf=1
            
           )

In [ ]:
rf.fit(x_train, y_train)

## Generate the Kernel Matrix

In [ ]:
# compute/get proximity matrix and visualize with seaborn heatmap


ref_map = rf.get_reference_map()
print(ref_map.shape)
# Show nnz per row (avg)
print(f"Average non-zeros per row: {ref_map.nnz / ref_map.shape[0]}")
print(type(ref_map))

leaf_mat = rf.apply(x_train)

for t in range(leaf_mat.shape[1]):
    _, counts = np.unique(leaf_mat[:, t], return_counts=True)
    print(
        t,
        "n_used_leaves=", len(counts),
        "avg_leaf_size=", counts.mean(),
        "max_leaf_size=", counts.max()
    )


prox = rf.get_kernel()

In [ ]:
# NNZ
print(f"Proximity matrix shape: {prox.shape}, NNZ: {prox.nnz}, Density: {prox.nnz / (prox.shape[0] * prox.shape[1])}")

# Visualize it

In [ ]:
prox_mat = prox.toarray()

# mask diagonal to emphasize off-diagonal proximities
mask = np.eye(prox_mat.shape[0], dtype=bool)
mask=None

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(prox_mat, ax=ax, cmap='rocket_r', vmin=0, vmax=prox_mat.max(),
            mask=mask,
            xticklabels=False, yticklabels=False, square=True, cbar_kws={'label': 'Proximity'})
ax.set_title(f'{kernel_method.upper()} Proximity Matrix')
plt.show()

# Check row sums
row_sums = prox_mat.sum(axis=1)
print("Row sums :")
print(row_sums)

# Check column sums
col_sums = prox_mat.sum(axis=0)
print("Column sums :")
print(col_sums)

# Show diagonal values
diag = prox.diagonal()
print("Diagonal values:")
print(diag)

# Show max off diagonal value
off_diag_max = prox_mat - np.diag(diag)
print("Max off-diagonal value:")
print(off_diag_max.max())

## Check extended kernel computation using a training subset

In [ ]:
# # selected_train_indices = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
# selected_train_indices = np.arange(x_train.shape[0])
# test_prox = rf.kernel_extend(x_test).toarray()
# # test_prox = rf.kernel_extend(x_test).toarray()
# fig, ax = plt.subplots(figsize=(10, 6))
# sns.heatmap(test_prox, ax=ax, cmap='rocket_r', vmin=0, vmax=test_prox.max(),
#             xticklabels=[f'train_{i}' for i in selected_train_indices],
#             yticklabels=False, cbar_kws={'label': 'Proximity'})
# ax.set_xlabel('Selected training indices')
# ax.set_ylabel('Test samples')
# ax.set_title(f'{kernel_method.upper()} Test-to-Train Proximity (shape={test_prox.shape})')
# plt.tight_layout()
# plt.show()

## Check Sum-to-One

In [ ]:
# check row sums (should sum to 1)
row_sums = np.sum(prox.toarray(), axis=1)

print("rows:", row_sums.shape[0])
print("min, max, mean:", row_sums.min(), row_sums.max(), row_sums.mean())

tol_r, tol_a = 1e-5, 1e-8
close_mask = np.isclose(row_sums, 1.0, rtol=tol_r, atol=tol_a)
print(f"rows ~1 within rtol={tol_r}, atol={tol_a}: {np.count_nonzero(close_mask)}/{len(row_sums)}")

bad_idx = np.where(~close_mask)[0]
if bad_idx.size:
    print("Example rows not summing to ~1 (index, sum):")
    for i in bad_idx[:10]:
        print(i, row_sums[i])
else:
    print("All rows sum approximately to 1.")

# Check new prox weighted predictions

In [ ]:
rf.kernel_predict(x_test)

In [ ]:
rf.predict_forest(x_test)

# Measures of Trust

### RF-ICE (Trust Scores)

In [ ]:
ice_scores = rf.get_instance_classification_expectation()
print(rf.trust_auc)

#### Accuracy Rejection Curve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

sns.lineplot(x=rf.trust_n_drop, y=rf.trust_accuracy_drop, ax=ax[0])
ax[0].set_xlabel('Number of Dropped Points')
ax[0].set_ylabel('Accuracy Drop')
ax[0].set_title('Accuracy Rejection Curve (RF-ICE)')
ax[0].set_ylim(0, None)

sns.scatterplot(x=x_train[:, 2], y=x_train[:, 3], hue=y_train, palette='Dark2',
                size=(np.max(rf.trust_scores) - rf.trust_scores) * 100 + 5, alpha=0.5, ax=ax[1])
ax[1].set_xlabel('Petal Length (normalized)')
ax[1].set_ylabel('Petal Width (normalized)')
ax[1].set_title('Scatter Plot of Training Data with 1 - Trust Scores')


### RF-ICE for Test Set

In [ ]:
trust_scores_test = rf.get_test_trust(x_test)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey = True)

sns.scatterplot(x=x_train[:, 2], y=x_train[:, 3], hue=y_train, palette='Dark2',
                size=(np.max(rf.trust_scores) - rf.trust_scores) * 100 + 5, alpha=0.5, ax=ax[0])
ax[0].set_xlabel('Petal Length (normalized)')
ax[0].set_ylabel('Petal Width (normalized)')
ax[0].set_title('Training Data with 1 - Trust Scores')

sns.scatterplot(x=x_test[:, 2], y=x_test[:, 3], hue=y_test, palette='Dark2',
                size=(np.max(rf.trust_scores_test) - rf.trust_scores_test) * 100 + 5, alpha=0.5, ax=ax[1])
ax[1].set_xlabel('Petal Length (normalized)')
ax[1].set_ylabel('Petal Width (normalized)')
ax[1].set_title('Test Data with 1 - Trust Scores')

In [ ]:
rf.get_nonconformity(k = 5, x_test = x_test, proximity_type = 'gap')

In [ ]:
rf.test_proximities.shape

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))


ax[0].scatter(x_train[:, 2], x_train[:, 3], c=y_train, cmap='Dark2', 
            s=rf.nonconformity_scores * 20 + 15, alpha=0.5)
ax[0].set_xlabel('Petal Length (normalized)')
ax[0].set_ylabel('Petal Width (normalized)')
ax[0].set_title('Scatter Plot of Training Data with Nonconformity Scores')


ax[1].scatter(x_test[:, 2], x_test[:, 3], c = y_test, cmap='Dark2', 
            s=rf.nonconformity_scores_test * 20 + 15, alpha=0.5)
ax[1].set_xlabel('Petal Length (normalized)')
ax[1].set_ylabel('Petal Width (normalized)')
ax[1].set_title('Scatter Plot of Training Data with Nonconformity Scores')

In [ ]:
sns.lineplot(x=rf.conformity_n_drop, y=rf.conformity_accuracy_drop)
plt.xlabel('Number of Dropped Points')
plt.ylabel('Accuracy Drop')
plt.title('Conformity Accuracy Rejection Curve')

In [ ]:
plt.plot(rf.nonconformity_scores)

In [ ]:
                # Updates the following attributes:
                # - `self.nonconformity_scores`
                # - `self.conformity_scores`
                # - `self.conformity_quantiles`
                # - `self.conformity_auc`
                # - `self.conformity_accuracy_drop`
                # - `self.conformity_n_drop`
                
                # If `x_test` is provided, also updates:
                # - `self.nonconformity_scores_test`
                # - `self.conformity_scores_test`
                # - `self.conformity_quantiles_test`